# Run the pipeline

Compile the Dataform pipeline from `main` on GitHub and run it in your project, then wait for it to finish. This does the same as `infra/scripts/run_pipeline.sh`.

The first `tofu apply` already runs the pipeline once. Run this notebook to rebuild everything, for example after you change the as-of date in your fork. The run rebuilds `serving.mc_simulated_pnl` for every scenario in `raw.stress_scenarios`.

In [ ]:
import time

import google.auth
from google.auth.transport.requests import AuthorizedSession

# The Dataform region from infra/variables.tf (var.region). Change it here if you changed it there.
REGION = "us-central1"

credentials, PROJECT_ID = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
session = AuthorizedSession(credentials)
LOCATION = f"https://dataform.googleapis.com/v1/projects/{PROJECT_ID}/locations/{REGION}"

# The pipeline repository is the one linked to GitHub. Notebooks and saved
# queries are Dataform repositories too, so filter on the git link.
repositories = session.get(f"{LOCATION}/repositories").json().get("repositories", [])
pipeline = next(r for r in repositories if "gitRemoteSettings" in r)
BASE = f"https://dataform.googleapis.com/v1/{pipeline['name']}"
RUNNER_SA = f"dataform-runner@{PROJECT_ID}.iam.gserviceaccount.com"
print(f"Project: {PROJECT_ID}, repository: {pipeline['name'].rsplit('/', 1)[-1]}")

## Compile

`defaultDatabase` sets the project the SQL runs in. Without it, Dataform uses `defaultProject` from `workflow_settings.yaml`.

In [ ]:
compilation = session.post(f"{BASE}/compilationResults", json={
    "gitCommitish": "main",
    "codeCompilationConfig": {"defaultDatabase": PROJECT_ID},
})
compilation.raise_for_status()
compilation = compilation.json()
errors = compilation.get("compilationErrors", [])
if errors:
    raise RuntimeError(errors)
print("Compiled:", compilation["name"].rsplit("/", 1)[-1])

## Run and wait

In [ ]:
invocation = session.post(f"{BASE}/workflowInvocations", json={
    "compilationResult": compilation["name"],
    "invocationConfig": {"serviceAccount": RUNNER_SA},
})
invocation.raise_for_status()
name = invocation.json()["name"]
print("Started:", name.rsplit("/", 1)[-1])

while True:
    state = session.get(f"https://dataform.googleapis.com/v1/{name}").json()["state"]
    if state not in ("RUNNING", "CANCELING"):
        break
    time.sleep(15)
print("Finished:", state)

If the run failed, list the actions that did not succeed:

In [ ]:
actions = session.get(f"https://dataform.googleapis.com/v1/{name}:query").json().get("workflowInvocationActions", [])
for action in actions:
    if action.get("state") not in ("SUCCEEDED", "SKIPPED"):
        print(action["target"]["name"], action.get("state"), action.get("failureReason", ""))